# Safe Haven Assets Post-COVID: An Empirical Assessment of Hedging Properties

**Course:** Empirical Methods in Finance  
**Authors:** *[Names]*  
**Date:** May 2026

---

## Abstract

*[To be written after results are finalized.]*

---

## 0. Imports & Setup

All packages installed below. `arch` provides GJR-GARCH and GARCH estimation.
`ruptures` implements the Bai-Perron dynamic-programming algorithm. `statsmodels`
provides the Markov-switching filter (Hamilton 1989) and time-series diagnostics.

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'arch', 'ruptures', 'scikit-learn'], capture_output=True)

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy import stats
from scipy.stats import jarque_bera, kstest, norm
from scipy.optimize import minimize

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

from arch import arch_model
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import ruptures as rpt

warnings.filterwarnings('ignore')
np.random.seed(42)

os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/tables',  exist_ok=True)

# ── Global parameters (one place to change everything) ────────────────────────
COVID_DATE   = pd.Timestamp('2020-03-11')   # WHO pandemic declaration
SPLIT_DATE   = '2020-03-01'                 # provisional pre/post split
DURATION_10Y = 8.5                           # modified duration for 10Y bond return proxy
TRIM         = 0.15                          # Andrews (1993) trimming fraction
SUPF_CV_5PCT = 8.85                          # Andrews 5% critical value, k=1 restriction
MIN_SEG_BP   = 252                           # Bai-Perron minimum segment = 1 year of daily obs
ROLL_WIN     = 252                           # rolling correlation window (1 year, daily)
OOS_CUTOFF   = '2020-01-01'                  # out-of-sample cutoff for volatility tournament

safe_havens = ['Gold', 'US10Y', 'USDCHF', 'USDJPY']
benchmarks  = ['SP500', 'Eurostoxx50']       # test vs BOTH equity benchmarks

#### justification cste TRIM
!!!!text pure IA!!!! 
Following Andrews (1993), we set the trimming fraction to 15% in the sup-F test applied in Step 4.2 to each DCC correlation series, meaning the test searches for breaks only within the central 70% of the sample. The value of 15% is not arbitrary: Andrews (1993) derived it as the practical optimum balancing the risk of boundary noise against the cost of shrinking the search region, and it has since become the standard convention in applied structural break testing, adopted notably by Bai and Perron (1998, 2003) in their multiple-break extension. In our application this choice has no material consequence, since the COVID-19 event of March 2020 falls well within the interior of our 1990–2026 sample, far from either trimmed boundary.

---

# 1. Data & Stylized Facts

## Step 1.1 — Data Loading & Sign Conventions

Six assets from `data/Data.xlsx`. We compute log-returns with the following sign
conventions so that **a positive value always represents a profit for the holder**
(or safe-haven appreciation):

| Asset | Transformation |
|---|---|
| S&P 500, Eurostoxx 50, Gold | $\ln(P_t / P_{t-1})$ |
| US 10Y yield | $-\Delta y_t \times D_{\text{mod}} / 100$ (duration approximation; positive = bond price rises) |
| USDCHF, USDJPY | $-\ln(P_t / P_{t-1})$ (CHF/JPY appreciation as positive) |

**Critical note on data loading:** we compute log-returns *before* any forward-filling
so that we only work with actual trading-day returns. Forward-filling prices before
computing returns would create artificial zero-return observations on non-trading days,
distorting GARCH estimates, ACF diagnostics, and rolling correlations. After computing
returns, we drop rows with any remaining missing value.

Returns are multiplied by 100 (percentage points) for numerical stability in GARCH
estimation. The provisional pre/post split uses **2020-03-01** and is later confirmed
endogenously by Bai-Perron.

In [ ]:
COLS = {
    'S&P500':             'SP500',
    'Eurostoxx 50':       'Eurostoxx50',
    'Gold':               'Gold',
    'US T 10-year Yield': 'US10Y_yield',   # raw yield level
    'USDCHF':             'USDCHF',
    'USDJPY':             'USDJPY',
}

# Load raw daily prices — no ffill yet, we want actual prices
raw = (pd.read_excel(Path("data/Data.xlsx"), index_col=0, parse_dates=True)
         [list(COLS.keys())]
         .rename(columns=COLS))

# ── Compute returns on actual prices BEFORE any filling ───────────────────────
returns = pd.DataFrame(index=raw.index)
returns['SP500']       = np.log(raw['SP500']       / raw['SP500'].shift(1))
returns['Eurostoxx50'] = np.log(raw['Eurostoxx50'] / raw['Eurostoxx50'].shift(1))
returns['Gold']        = np.log(raw['Gold']         / raw['Gold'].shift(1))

# Duration approximation: ΔP/P ≈ -D_mod × Δy_decimal
# DURATION_10Y = 8.5 years is a central estimate; see limitations in report.
returns['US10Y']  = -raw['US10Y_yield'].diff() * DURATION_10Y / 100

# Negate FX returns: falling USDCHF/USDJPY = foreign currency appreciating = safe-haven response
returns['USDCHF'] = -np.log(raw['USDCHF'] / raw['USDCHF'].shift(1))
returns['USDJPY'] = -np.log(raw['USDJPY'] / raw['USDJPY'].shift(1))

# Drop rows with any NaN (actual missing trading days, first row after shift)
returns = returns.dropna() * 100   # scale to percentage points

# ── Pre / post split ──────────────────────────────────────────────────────────
ret_pre  = returns.loc[:SPLIT_DATE]
ret_post = returns.loc[SPLIT_DATE:]

print(f"Full sample : {returns.index[0].date()} → {returns.index[-1].date()} ({len(returns):,} obs)")
print(f"Pre-COVID   : {ret_pre.index[0].date()} → {ret_pre.index[-1].date()} ({len(ret_pre):,} obs)")
print(f"Post-COVID  : {ret_post.index[0].date()} → {ret_post.index[-1].date()} ({len(ret_post):,} obs)")
print(f"\nAll assets  : {list(returns.columns)}")

## Step 1.2 — Descriptive Statistics

Annualized mean and volatility, skewness, excess kurtosis, max drawdown, and four
diagnostic tests:

- **Jarque-Bera / Kolmogorov-Smirnov**: normality — we expect rejection, motivating the
  Student-t distribution in the GJR-GARCH.
- **ADF + KPSS**: stationarity — we need $p_{\text{ADF}} < 0.05$ and $p_{\text{KPSS}} > 0.05$
  to confirm the series are $I(0)$, a prerequisite for valid OLS and GARCH inference.
- **Ljung-Box on squared returns**: ARCH effects — a very small p-value means squared
  returns are autocorrelated, which is the empirical motivation for GARCH.

In [ ]:
def stylized_facts(df, label):
    stats_df = pd.DataFrame(index=df.columns)
    stats_df['Mean (ann %)'] = df.mean() * 252
    stats_df['Vol (ann %)']  = df.std()  * np.sqrt(252)
    stats_df['Skewness']     = df.skew()
    stats_df['Excess kurt']  = df.kurtosis()
    stats_df['Max DD %']     = (
        (1 + df / 100).cumprod() / (1 + df / 100).cumprod().cummax() - 1
    ).min() * 100

    jb, ks_p, adf_p, kpss_p, lb_p = [], [], [], [], []
    for col in df.columns:
        s = df[col].dropna()
        jb.append(jarque_bera(s)[1])
        ks_p.append(kstest(s, 'norm', args=(s.mean(), s.std()))[1])
        adf_p.append(adfuller(s)[1])
        kpss_p.append(kpss(s, regression='c', nlags='auto')[1])
        lb_p.append(acorr_ljungbox(s**2, lags=[10], return_df=True)['lb_pvalue'].iloc[0])

    stats_df['JB p']   = jb
    stats_df['KS p']   = ks_p
    stats_df['ADF p']  = adf_p
    stats_df['KPSS p'] = kpss_p
    stats_df['LB² p']  = lb_p
    print(f"\n{'═'*60}\n{label}\n{'═'*60}")
    return stats_df.round(4)

table1_pre  = stylized_facts(ret_pre,  "Pre-COVID")
display(table1_pre)
table1_post = stylized_facts(ret_post, "Post-COVID")
display(table1_post)

table1_pre.to_csv('outputs/tables/table1_pre.csv')
table1_post.to_csv('outputs/tables/table1_post.csv')

## Step 1.3 — ACF of Returns and Squared Returns

If the ACF of $r_t$ shows little autocorrelation but the ACF of $r_t^2$ decays
slowly, this is the signature of **volatility clustering**: large moves tend to
follow large moves regardless of sign. This visual evidence motivates the GARCH
family. Note: even if we knew $\sigma_t^2$ exactly, $r_t^2$ would fall within
$\pm 50\%$ of the true variance only about 26% of the time (under $\chi^2(1)$),
which is why a model-based filter is needed.

In [ ]:
fig, axes = plt.subplots(2, len(returns.columns), figsize=(20, 6))
for i, col in enumerate(returns.columns):
    plot_acf(returns[col],      ax=axes[0, i], lags=30, title=col,     auto_ylims=True)
    plot_acf(returns[col] ** 2, ax=axes[1, i], lags=30, title=f"{col}²", auto_ylims=True)
axes[0, 0].set_ylabel('Returns ACF')
axes[1, 0].set_ylabel('Squared returns ACF')
plt.suptitle('ACF of Returns and Squared Returns (30 lags)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/figures/acf.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 2. Univariate Volatility & Out-of-Sample Forecasting

We compare three volatility models on an out-of-sample period starting January 2020:

1. **EWMA** ($\varphi = 0.94$) — RiskMetrics; no parameters to estimate, fully recursive.
2. **HAR** — OLS regression of realized variance on daily, weekly, and monthly averages.
3. **GJR-GARCH(1,1)-t** — captures clustering, leverage effect ($\gamma > 0$), and fat tails.

All three are trained exclusively on pre-2020 data and evaluated on post-2020 data.
Patton (2011) robust loss functions are used: MSE and QLIKE. The Diebold-Mariano
test provides a formal test of equal predictive accuracy.

## Step 2.1 — Model Definitions

### EWMA
$$\sigma_t^2 = (1-\varphi)\,r_{t-1}^2 + \varphi\,\sigma_{t-1}^2, \quad \varphi=0.94$$

### HAR
$$RV_{t+1} = c + \beta_d RV_t^{(d)} + \beta_w RV_t^{(w)} + \beta_m RV_t^{(m)} + u_{t+1}$$

### GJR-GARCH(1,1)-t
$$\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \gamma\varepsilon_{t-1}^2\mathbf{1}_{\{\varepsilon_{t-1}<0\}} + \beta\sigma_{t-1}^2, \quad z_t\sim t_\nu$$

A significant positive $\gamma$ is the empirical signature of the leverage effect.

In [ ]:
# ── EWMA ──────────────────────────────────────────────────────────────────────
def ewma_variance(series, phi=0.94):
    var = np.zeros(len(series))
    var[0] = series.iloc[:30].var()
    for t in range(1, len(series)):
        var[t] = (1 - phi) * series.iloc[t-1]**2 + phi * var[t-1]
    return pd.Series(var, index=series.index)

# ── HAR ───────────────────────────────────────────────────────────────────────
def har_features(series):
    rv = series ** 2
    return pd.DataFrame({
        'RV':   rv,
        'RV_d': rv.shift(1),
        'RV_w': rv.shift(1).rolling(5).mean(),
        'RV_m': rv.shift(1).rolling(22).mean(),
    }).dropna()

def fit_har(series_train):
    df = har_features(series_train)
    X  = sm.add_constant(df[['RV_d', 'RV_w', 'RV_m']])
    return sm.OLS(df['RV'], X).fit()

def predict_har(model, series_full):
    df = har_features(series_full)
    X  = sm.add_constant(df[['RV_d', 'RV_w', 'RV_m']])
    return pd.Series(model.predict(X), index=df.index)

# ── GJR-GARCH ─────────────────────────────────────────────────────────────────
def fit_gjr(series):
    return arch_model(series, vol='GARCH', p=1, o=1, q=1, dist='studentst').fit(disp='off')

## Step 2.2 — Estimate GJR-GARCH: Full Sample and Pre/Post

Two sets of fits. The **full-sample** conditional volatility feeds the DCC filter in
Block 3. The **pre/post** fits are compared in Table 2 — a change in $\gamma$, $\nu$,
or persistence across the COVID break is motivational evidence that safe-haven volatility
dynamics shifted.

In [ ]:
# Full-sample fits (conditional vol used in DCC)
garch_full = {col: fit_gjr(returns[col]) for col in returns.columns}
# Pre / post fits (for parameter comparison)
garch_pre  = {col: fit_gjr(ret_pre[col])  for col in returns.columns}
garch_post = {col: fit_gjr(ret_post[col]) for col in returns.columns}

def garch_summary(model_dict, label):
    rows = []
    for col, res in model_dict.items():
        p = res.params
        rows.append({
            'omega':       p['omega'],
            'alpha':       p['alpha[1]'],
            'gamma':       p['gamma[1]'],
            'beta':        p['beta[1]'],
            'nu':          p['nu'],
            'persistence': p['alpha[1]'] + 0.5*p['gamma[1]'] + p['beta[1]'],
        })
    df = pd.DataFrame(rows, index=model_dict.keys()).round(4)
    df.columns = pd.MultiIndex.from_product([[label], df.columns])
    return df

table2 = pd.concat([
    garch_summary(garch_pre,  'Pre-COVID'),
    garch_summary(garch_post, 'Post-COVID')
], axis=1)
display(table2)
table2.to_csv('outputs/tables/table2_gjr_params.csv')

# ── Conditional volatility plot ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, returns.columns):
    ax.plot(garch_full[col].conditional_volatility * np.sqrt(252),
            linewidth=0.7, color='steelblue')
    ax.axvline(COVID_DATE, color='crimson', linestyle='--', linewidth=1.2)
    ax.set_title(col)
    ax.set_ylabel('Annualised cond. vol (%)')
plt.suptitle('GJR-GARCH(1,1)-t — Conditional Volatility (full sample)', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/conditional_vol.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2.3 — GARCH Diagnostics

Ljung-Box on squared standardized residuals $(\hat{z}_t^2 = (\hat{\varepsilon}_t / \hat{\sigma}_t)^2)$
at lag 10. A p-value above 0.05 means the GARCH has absorbed the ARCH effects — the
model is well specified.

In [ ]:
print("Ljung-Box on squared standardized residuals (p > 0.05 = well specified):")
for col, res in garch_full.items():
    z2 = (res.resid / res.conditional_volatility).dropna() ** 2
    p  = acorr_ljungbox(z2, lags=[10], return_df=True)['lb_pvalue'].iloc[0]
    flag = '✓' if p > 0.05 else '✗ ARCH remains'
    print(f"  {col:12s}  p = {p:.4f}  {flag}")

## Step 2.4 — Out-of-Sample Forecasting Tournament

All models trained on data before `2020-01-01`, evaluated on post-2020 observations.

Patton (2011) robust loss functions:
$$L^{\text{MSE}}_t = (r_t^2 - \hat{\sigma}_t^2)^2, \qquad
L^{\text{QLIKE}}_t = \frac{r_t^2}{\hat{\sigma}_t^2} - \ln\!\left(\frac{r_t^2}{\hat{\sigma}_t^2}\right) - 1$$

The Diebold-Mariano statistic tests $H_0$: equal predictive accuracy. A negative DM
means model A beats model B; a small p-value rejects equal accuracy.

In [ ]:
def qlike(actual, forecast):
    actual = np.maximum(actual, 1e-8)
    return actual / forecast - np.log(actual / forecast) - 1

def mse_loss(actual, forecast):
    return (actual - forecast) ** 2

def diebold_mariano(la, lb):
    d  = np.asarray(la) - np.asarray(lb)
    dm = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    return dm, 2 * (1 - norm.cdf(abs(dm)))

# GJR-GARCH: fit on pre-OOS data only, then fix params and filter on full sample
garch_pre_oos = {
    col: arch_model(returns[col], vol='GARCH', p=1, o=1, q=1, dist='studentst')
         .fit(last_obs=OOS_CUTOFF, disp='off')
    for col in returns.columns
}
garch_fc_full = {
    col: arch_model(returns[col], vol='GARCH', p=1, o=1, q=1, dist='studentst')
         .fix(garch_pre_oos[col].params).conditional_volatility
    for col in returns.columns
}
har_oos = {col: fit_har(returns[col].loc[:OOS_CUTOFF]) for col in returns.columns}

# ── OOS comparison table ──────────────────────────────────────────────────────
rows = []
for col in returns.columns:
    actual_var = (returns[col] ** 2).loc[OOS_CUTOFF:]
    garch_fc   = (garch_fc_full[col] ** 2).loc[OOS_CUTOFF:]
    ewma_fc    = ewma_variance(returns[col]).loc[OOS_CUTOFF:]
    har_fc     = predict_har(har_oos[col], returns[col]).loc[OOS_CUTOFF:]

    df = pd.DataFrame({'A': actual_var, 'GARCH': garch_fc,
                       'EWMA': ewma_fc, 'HAR': har_fc}).dropna()
    rows.append({
        'Asset':       col,
        'MSE GARCH':   mse_loss(df['A'], df['GARCH']).mean(),
        'MSE EWMA':    mse_loss(df['A'], df['EWMA']).mean(),
        'MSE HAR':     mse_loss(df['A'], df['HAR']).mean(),
        'QLIKE GARCH': qlike(df['A'], df['GARCH']).mean(),
        'QLIKE EWMA':  qlike(df['A'], df['EWMA']).mean(),
        'QLIKE HAR':   qlike(df['A'], df['HAR']).mean(),
    })

table3 = pd.DataFrame(rows).set_index('Asset').round(4)
display(table3)
table3.to_csv('outputs/tables/table3_oos.csv')

# ── Diebold-Mariano ───────────────────────────────────────────────────────────
pairs  = [('GARCH', 'EWMA'), ('GARCH', 'HAR'), ('EWMA', 'HAR')]
dm_rows = []
for col in returns.columns:
    actual_var = (returns[col] ** 2).loc[OOS_CUTOFF:]
    fc = {
        'GARCH': (garch_fc_full[col] ** 2).loc[OOS_CUTOFF:],
        'EWMA':  ewma_variance(returns[col]).loc[OOS_CUTOFF:],
        'HAR':   predict_har(har_oos[col], returns[col]).loc[OOS_CUTOFF:],
    }
    df = pd.DataFrame({'A': actual_var, **fc}).dropna()
    for a, b in pairs:
        la   = qlike(df['A'], df[a])
        lb   = qlike(df['A'], df[b])
        stat, p = diebold_mariano(la, lb)
        dm_rows.append({'Asset': col, 'Pair': f'{a} vs {b}',
                        'DM stat': round(stat, 3), 'p-value': round(p, 4)})

dm_table = pd.DataFrame(dm_rows).set_index(['Asset', 'Pair'])
display(dm_table)
dm_table.to_csv('outputs/tables/table3_dm.csv')

---

# 3. Multivariate Dependence

## Step 3.1 — DCC (EWMA-Based, Two-Step)

We follow Engle (2002) using an EWMA-based correlation recursion with $\lambda = 0.97$
(half-life $\approx 23$ days, appropriate for correlation estimation on daily data).
The inputs $u_{i,t}$ are the standardized residuals from the full-sample GJR-GARCH
estimated in Step 2.2, which have had each asset's individual volatility dynamics removed.

The DCC correlation $\rho_t^{ij}$ is **$\mathcal{F}_{t-1}$-measurable** — fully
determined by past information before observing $r_t$ — making it a genuine
conditional estimate rather than the unconditional rolling average in Step 3.3.

We test each safe haven against **both equity benchmarks** (S&P 500 and Eurostoxx 50)
to examine whether the hedge breakdown is specific to US equities or global.

In [ ]:
# Standardized residuals from full-sample GJR-GARCH
std_resid = pd.DataFrame({
    col: garch_full[col].resid / garch_full[col].conditional_volatility
    for col in returns.columns
}).dropna()

def dcc_ewma(u, lam=0.97):
    u_arr = u.values
    T, N  = u_arr.shape
    Q     = np.cov(u_arr.T)
    R     = np.zeros((T, N, N))
    R[0]  = np.corrcoef(u_arr.T)
    for t in range(1, T):
        ut_1 = u_arr[t-1].reshape(-1, 1)
        Q    = (1 - lam) * (ut_1 @ ut_1.T) + lam * Q
        d_inv = np.diag(1 / np.sqrt(np.diag(Q)))
        R[t]  = d_inv @ Q @ d_inv
    return R

R_dyn = dcc_ewma(std_resid)
col_list = list(returns.columns)

# Extract DCC correlation for each safe-haven / benchmark pair
dcc_corr_all = {}
for sh in safe_havens:
    j_sh = col_list.index(sh)
    for bm in benchmarks:
        j_bm = col_list.index(bm)
        key  = f"{sh}_vs_{bm}"
        dcc_corr_all[key] = pd.Series(
            R_dyn[:, j_sh, j_bm], index=std_resid.index, name=key
        )

print(f"DCC computed for {len(dcc_corr_all)} pairs: {list(dcc_corr_all.keys())}")

## Step 3.2 — Conditional Beta Decomposition

For each safe haven $j$ and benchmark $m$:

$$\beta_{j,t} = \rho_{j,m,t} \times \frac{\sigma_{j,t}}{\sigma_{m,t}}$$

Decomposing $\beta$ into a **correlation channel** ($\rho$) and a **relative-volatility
channel** ($\sigma_j/\sigma_m$) identifies whether a post-COVID change in the hedge
ratio came from increased co-movement or from a shift in relative volatility profiles.
We display this decomposition for all safe havens against S&P 500.

In [ ]:
# Compute conditional beta for all pairs
dcc_beta_all = {}
for sh in safe_havens:
    sigma_sh = garch_full[sh].conditional_volatility.loc[std_resid.index]
    for bm in benchmarks:
        sigma_bm = garch_full[bm].conditional_volatility.loc[std_resid.index]
        key = f"{sh}_vs_{bm}"
        dcc_beta_all[key] = dcc_corr_all[key] * (sigma_sh / sigma_bm)

# ── Figure: beta decomposition vs SP500 ──────────────────────────────────────
fig, axes = plt.subplots(4, 3, figsize=(17, 12), sharex=True)
for i, sh in enumerate(safe_havens):
    key = f"{sh}_vs_SP500"
    rho_t    = dcc_corr_all[key]
    sigma_sh = garch_full[sh].conditional_volatility.loc[std_resid.index]
    sigma_sp = garch_full['SP500'].conditional_volatility.loc[std_resid.index]

    for ax, (data, title, color) in zip(axes[i], [
        (rho_t,               'DCC ρ_t',             'darkblue'),
        (sigma_sh / sigma_sp, 'σ_sh / σ_SP500',      'seagreen'),
        (dcc_beta_all[key],   'β_t = ρ_t × (σ_sh/σ_m)', 'purple'),
    ]):
        ax.plot(data.index, data.values, linewidth=0.7, color=color)
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
        ax.axvline(COVID_DATE, color='crimson', linestyle='--', linewidth=1.2)
        ax.set_ylabel(sh, fontsize=9)
        if i == 0: ax.set_title(title, fontsize=10)

plt.suptitle('Conditional Beta Decomposition: β_t = ρ_t × (σ_sh / σ_market)\n'
             'vs S&P 500  |  Red dashed = COVID-19', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/conditional_beta.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3.3 — Rolling 252-Day Correlation

A non-parametric counterpart to the DCC. The rolling correlation uses a fixed
252-day window (one trading year) and weights all observations equally — unlike
the DCC, which weights recent observations more heavily. If both series flag the
same structural break in Section 4, the result is robust to the choice of estimator.

We compute rolling correlations for all eight safe-haven / benchmark pairs.

In [ ]:
rolling_corr_all = {}
for sh in safe_havens:
    for bm in benchmarks:
        key = f"{sh}_vs_{bm}"
        rolling_corr_all[key] = (
            returns[sh].rolling(ROLL_WIN).corr(returns[bm]).dropna()
        )

# ── Figure: DCC vs rolling for all 8 pairs ────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
for i, sh in enumerate(safe_havens):
    for j, bm in enumerate(benchmarks):
        key = f"{sh}_vs_{bm}"
        ax  = axes[i, j]
        ax.plot(dcc_corr_all[key].index, dcc_corr_all[key].values,
                color='darkblue', linewidth=0.7, label='DCC', alpha=0.85)
        ax.plot(rolling_corr_all[key].index, rolling_corr_all[key].values,
                color='lightsteelblue', linewidth=1.0, label='Rolling 252d')
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
        ax.axvline(COVID_DATE, color='crimson', linestyle='--',
                   linewidth=1.2, label='COVID-19')
        ax.fill_between(dcc_corr_all[key].index, dcc_corr_all[key].values, 0,
                        where=(dcc_corr_all[key].values < 0),
                        alpha=0.10, color='green', label='Hedge region')
        ax.set_ylim(-0.85, 0.85)
        ax.set_title(f'{sh} vs {bm}', fontsize=9)
        if i == 0 and j == 0:
            ax.legend(fontsize=7)

plt.suptitle('DCC vs Rolling 252-Day Correlation — All 8 Safe-Haven / Benchmark Pairs',
             fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/correlations.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 4. Structural Breaks & Regimes

## Step 4.1 — Rolling PCA

The fraction of variance explained by the first principal component over a rolling
252-day window:

$$V^{PC1}_t = \frac{\lambda_{1,t}}{\sum_k \lambda_{k,t}}$$

A rising $V^{PC1}$ means a single common factor dominates cross-asset returns —
diversification weakens. If this measure spikes and stays elevated after COVID, it
provides additional evidence of a correlation regime shift. We also compare PC1
loadings pre vs post to see whether safe havens have moved closer to the dominant
equity factor.

In [ ]:
window = ROLL_WIN
v_pc1  = pd.Series(index=returns.index, dtype=float)

for t in range(window, len(returns)):
    chunk  = returns.iloc[t - window:t]
    scaled = StandardScaler().fit_transform(chunk)
    pca    = PCA(n_components=chunk.shape[1]).fit(scaled)
    v_pc1.iloc[t] = pca.explained_variance_ratio_[0]

v_pc1 = v_pc1.dropna()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(v_pc1.index, v_pc1 * 100, linewidth=0.8, color='steelblue')
ax.axvline(COVID_DATE, color='crimson', linestyle='--', linewidth=1.5, label='COVID-19')
ax.set_ylabel('Variance explained by PC1 (%)')
ax.set_title('Rolling PCA — Share of Variance Explained by First Principal Component\n'
             '(Rising = diversification weakening, assets moving together)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/rolling_vpc1.png', dpi=150, bbox_inches='tight')
plt.show()

# PC1 loadings pre / post
def pca_loadings(df):
    scaled = StandardScaler().fit_transform(df)
    pca    = PCA(n_components=df.shape[1]).fit(scaled)
    return pd.Series(pca.components_[0], index=df.columns)

loadings = pd.DataFrame({
    'Pre-COVID':  pca_loadings(ret_pre),
    'Post-COVID': pca_loadings(ret_post),
}).round(4)
print("PC1 loadings pre vs post COVID:")
display(loadings)
loadings.to_csv('outputs/tables/pca_loadings.csv')

## Step 4.2 — Structural Break Tests: sup-F then Bai-Perron

We apply two-stage structural break testing to each DCC correlation series.

**Stage 1 — sup-F (Quandt-Andrews test).** This preliminary test asks whether *any*
evidence of at least one break exists, without specifying the date or number. The
statistic is the supremum of Chow F-statistics across all admissible candidate dates
(excluding 15% from each end). Its critical values are non-standard (Andrews 1993)
because they account for the search: using ordinary F tables after searching would
systematically over-reject. For $k=1$ restriction (mean shift) and 15% trimming,
the 5% critical value is approximately 8.85. If sup-F fails to reject, we stop and
report no statistical evidence of a break for that pair.

**Stage 2 — Bai-Perron (1998, 2003).** For pairs where sup-F rejects, we apply
dynamic programming to find the globally optimal break dates by minimizing total SSR
across all segments simultaneously. The number of breaks is selected endogenously
by comparing total SSR for $m = 1, 2, 3$: we add a break only if it reduces SSR by
more than 15% relative to $m-1$ breaks. This is more rigorous than imposing a fixed
number in advance.

In [ ]:
# ── sup-F test (Quandt-Andrews) ───────────────────────────────────────────────
def supF_test(series, trim=TRIM):
    y   = series.dropna().values
    T   = len(y)
    lo  = int(trim * T)
    hi  = T - int(trim * T)
    SSR_full = np.sum((y - y.mean())**2)

    F_vals = {}
    for t in range(lo, hi):
        y1, y2 = y[:t], y[t:]
        SSR1 = np.sum((y1 - y1.mean())**2)
        SSR2 = np.sum((y2 - y2.mean())**2)
        num  = (SSR_full - SSR1 - SSR2) / 1
        den  = (SSR1 + SSR2) / (T - 2)
        F_vals[t] = num / den if den > 0 else 0.0

    sup_F     = max(F_vals.values())
    t_star    = max(F_vals, key=F_vals.get)
    all_dates = series.dropna().index
    F_series  = pd.Series(
        {all_dates[t]: v for t, v in F_vals.items()}, name='F_stat'
    )
    return sup_F, t_star, all_dates[t_star], F_series


supF_results   = {}
bp_results     = {}

print("=" * 70)
print(f"STAGE 1 — sup-F  (Andrews 1993  5% CV = {SUPF_CV_5PCT})")
print("=" * 70)

PAIRS = [f"{sh}_vs_{bm}" for sh in safe_havens for bm in benchmarks]

for key in PAIRS:
    series = dcc_corr_all[key]
    sup_F, t_idx, date_star, F_series = supF_test(series)
    reject = sup_F > SUPF_CV_5PCT
    supF_results[key] = {'sup_F': sup_F, 'date': date_star, 'reject': reject}

    flag = '→ REJECT ✓  (break exists, proceed to Bai-Perron)' if reject else '→ fail to reject'
    print(f"
{key}:")
    print(f"  sup-F = {sup_F:.2f}   {flag}")
    print(f"  sup-F estimated break: {date_star.date()}")

    if reject:
        # ── STAGE 2: Bai-Perron ───────────────────────────────────────────────
        y_bp  = series.dropna().values.reshape(-1, 1)
        idx_  = series.dropna().index
        model_bp = rpt.Dynp(model="l2", min_size=MIN_SEG_BP, jump=1).fit(y_bp)

        def total_ssr(bkps):
            segs = [0] + bkps
            ssr  = 0
            for k in range(len(segs) - 1):
                seg = y_bp[segs[k]:segs[k+1]]
                ssr += float(np.sum((seg - seg.mean())**2))
            return ssr

        ssr_by_m  = {0: total_ssr([len(y_bp)])}
        bkps_by_m = {}
        for m in [1, 2, 3]:
            bkps_m       = model_bp.predict(n_bkps=m)
            ssr_by_m[m]  = total_ssr(bkps_m)
            bkps_by_m[m] = bkps_m[:-1]   # drop sentinel

        # Sequential selection: keep adding breaks while SSR drops > 15%
        optimal_m = 0
        for m in [1, 2, 3]:
            reduction = (ssr_by_m[m-1] - ssr_by_m[m]) / ssr_by_m[m-1]
            if reduction > 0.15:
                optimal_m = m

        break_dates = [idx_[b] for b in bkps_by_m.get(optimal_m, [])]
        bp_results[key] = {'optimal_m': optimal_m, 'break_dates': break_dates,
                           'ssr_by_m': ssr_by_m}

        print(f"  Bai-Perron: optimal m = {optimal_m}")
        for m in range(1, 4):
            red = (ssr_by_m[m-1] - ssr_by_m[m]) / ssr_by_m[m-1] * 100
            print(f"    m={m}: SSR = {ssr_by_m[m]:.4f}  (−{red:.1f}% vs m−1)")
        for d in break_dates:
            print(f"  → Break detected: {d.date()}")
    else:
        bp_results[key] = {'optimal_m': 0, 'break_dates': [], 'ssr_by_m': {}}

# Save
bp_table = pd.DataFrame({
    k: [str(d.date()) for d in v['break_dates']] +
       [''] * (3 - len(v['break_dates']))
    for k, v in bp_results.items()
}, index=['Break 1', 'Break 2', 'Break 3']).T
bp_table.to_csv('outputs/tables/bai_perron_breaks.csv')
display(bp_table)

## Step 4.3 — Markov-Switching (2-State)

Unlike Bai-Perron, which detects permanent breaks, Markov switching models
**recurring** regimes — states the market can revisit multiple times. We fit a
two-state model with switching mean and switching variance on the rolling correlation
series (which provides a smoother signal for regime detection). State 1 is the
*hedging* regime (low or negative correlation) and State 2 is the *safe-haven
breakdown* regime (high positive correlation).

The Hamilton filter (1989) delivers the filtered probability $P(S_t = 2\mid \mathcal{F}_{t-1})$
in real time, which is directly actionable for a portfolio manager. The expected
duration of each regime is $D_i = 1/(1-p_{ii})$: a longer post-COVID duration for
the bad-hedge regime would indicate the market is spending structurally more time in it.

A short comment on TAR/SETAR: Lecture 12 introduced threshold AR models as an
alternative, with regime transitions driven by an observable threshold. Their key
limitation — noted explicitly in class — is that regime persistence is tied to the
autocorrelation of the threshold variable, providing no free parameter to model
duration independently. This is why we use Markov switching.

In [ ]:
def fit_ms(series):
    return MarkovRegression(
        series.dropna(), k_regimes=2, trend='c', switching_variance=True
    ).fit(disp=False, maxiter=300)

# Fit on rolling correlation (smoother signal)
ms_full = {}
ms_pre  = {}
ms_post = {}

for sh in safe_havens:
    # Use the SP500 pair as the primary series for regime detection
    key = f"{sh}_vs_SP500"
    s   = rolling_corr_all[key]
    try:
        ms_full[sh] = fit_ms(s)
        ms_pre[sh]  = fit_ms(s.loc[:SPLIT_DATE])
        ms_post[sh] = fit_ms(s.loc[SPLIT_DATE:])
    except Exception as e:
        print(f"{sh}: MS failed — {e}")

# ── Table 4: MS parameters pre / post ─────────────────────────────────────────
def ms_row(res, asset, period):
    try:
        # regime_transition gives p[i,j] — rows = from, cols = to
        p11 = float(res.regime_transition[0, 0, 0])
        p22 = float(res.regime_transition[1, 1, 0])
    except Exception:
        # fallback: read from params directly
        p11 = res.params.get('p[0->0]', np.nan)
        p22 = res.params.get('p[1->1]', np.nan)
    return {
        'Asset': asset, 'Period': period,
        'p11': round(p11, 4), 'p22': round(p22, 4),
        'D1 (days)': round(1 / (1 - p11), 1) if p11 < 1 else np.nan,
        'D2 (days)': round(1 / (1 - p22), 1) if p22 < 1 else np.nan,
    }

table4 = pd.DataFrame(
    [ms_row(ms_pre[c],  c, 'Pre-COVID')  for c in safe_havens if c in ms_pre]  +
    [ms_row(ms_post[c], c, 'Post-COVID') for c in safe_havens if c in ms_post]
).set_index(['Asset', 'Period']).round(4)
display(table4)
table4.to_csv('outputs/tables/table4_ms_params.csv')

# ── Figure: smoothed regime-2 probabilities ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, sh in zip(axes.flat, safe_havens):
    if sh not in ms_full:
        continue
    try:
        probs = ms_full[sh].smoothed_marginal_probabilities.iloc[:, 1]
    except Exception:
        probs = ms_full[sh].smoothed_marginal_probabilities[1]
    ax.fill_between(probs.index, probs.values,
                    alpha=0.4, color='crimson', label='P(breakdown regime)')
    ax.axvline(COVID_DATE, color='black', linestyle='--', linewidth=1.2)
    ax.set_title(f'{sh} vs SP500 — Regime-2 probability', fontsize=10)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.suptitle('Smoothed Markov-Switching Regime Probabilities\n'
             'Regime 2 = safe-haven breakdown (high correlation)', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/smoothed_probs.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4.4 — Wald Test: Did the Regime Structure Itself Change?

Structural break tests tell us *when* the correlation level shifted; Markov switching
tells us about recurring regime dynamics. But neither alone answers whether COVID
changed the fundamental nature of the regimes — not just which one the market visited,
but what each regime looks like.

We bridge the two analyses with a Wald test. Using the Bai-Perron break date as the
split point, we estimate the full Markov switching parameter vector
$\hat{\theta} = (\mu_1, \mu_2, \sigma_1, \sigma_2, p_{11}, p_{22})$ separately on
each subsample and test $H_0: \theta_{\text{pre}} = \theta_{\text{post}}$.

Since the two subsamples are independent, the variance of the parameter difference is
the sum of the two covariance matrices, and the Wald statistic
$W = (\hat{\theta}_{\text{pre}} - \hat{\theta}_{\text{post}})^\top
     [\hat{V}_{\text{pre}} + \hat{V}_{\text{post}}]^{-1}
     (\hat{\theta}_{\text{pre}} - \hat{\theta}_{\text{post}}) \sim \chi^2(k)$
under $H_0$, where $k$ is the number of parameters.

**Interpretation:**
- If we *reject* $H_0$: the regime dynamics themselves changed post-COVID — the bear
  regime has a different volatility, mean, or persistence. This is a structural change
  in the full sense.
- If we *fail to reject* $H_0$: the same two regimes existed before and after COVID;
  what changed was only the frequency or duration of visits to the bad-hedge regime.

In [ ]:
print("=" * 70)
print("WALD TEST: H₀ — Markov switching parameters stable across break date")
print("=" * 70)

wald_results = {}

for sh in safe_havens:
    key    = f"{sh}_vs_SP500"
    bp     = bp_results.get(key, {})
    breaks = bp.get('break_dates', [])

    if not breaks:
        print(f"\n{sh}: no Bai-Perron break detected — Wald test skipped")
        continue

    split  = breaks[0]   # use first (primary) break date
    s_full = rolling_corr_all[key]
    s_pre  = s_full.loc[:split]
    s_post = s_full.loc[split:]

    print(f"\n{sh}  (split at {split.date()}, "
          f"pre={len(s_pre)} obs, post={len(s_post)} obs)")

    if len(s_pre) < 60 or len(s_post) < 60:
        print(f"  Insufficient observations — skipping")
        continue

    try:
        ms_pre_wald  = fit_ms(s_pre)
        ms_post_wald = fit_ms(s_post)

        theta_pre  = ms_pre_wald.params.values
        theta_post = ms_post_wald.params.values
        diff       = theta_pre - theta_post

        var_pre  = np.diag(ms_pre_wald.cov_params().values)
        var_post = np.diag(ms_post_wald.cov_params().values)
        var_diff = var_pre + var_post

        W  = float(np.sum(diff**2 / (var_diff + 1e-12)))
        k  = len(diff)
        pv = 1 - stats.chi2.cdf(W, df=k)

        print(f"  Wald W = {W:.2f},  k = {k},  p = {pv:.4f}", end='  ')
        if pv < 0.05:
            print("→ REJECT H₀: regime structure changed — structural break in regime dynamics")
        else:
            print("→ fail to reject H₀: same regime dynamics, "
                  "market spent more time in bad-hedge regime post-COVID")

        # Compare expected duration of bad-hedge regime
        def get_p22(res):
            try:    return float(res.regime_transition[1, 1, 0])
            except: return res.params.get('p[1->1]', np.nan)

        p22_pre  = get_p22(ms_pre_wald)
        p22_post = get_p22(ms_post_wald)
        print(f"  E[duration bad regime]: pre = {1/(1-p22_pre):.1f} days, "
              f"post = {1/(1-p22_post):.1f} days")

        wald_results[sh] = {'W': W, 'k': k, 'p': pv,
                            'D_bad_pre': 1/(1-p22_pre),
                            'D_bad_post': 1/(1-p22_post)}

    except Exception as e:
        print(f"  Estimation failed: {e}")

wald_table = pd.DataFrame(wald_results).T.round(3)
display(wald_table)
wald_table.to_csv('outputs/tables/wald_ms.csv')

---

# 5. Portfolio Construction

We translate empirical results into portfolio terms — the dimension Ielpo explicitly
grades on decision relevance. Two complementary approaches:

- **ERC** (Equal Risk Contribution): no expected returns required; shows how the
  risk-balanced allocation should have shifted pre vs post COVID.
- **Markowitz with constant $\mu$**: minimum-variance optimization with target return,
  long-only, weights $\leq 70\%$, and $\mu$ held constant across regimes to isolate
  the pure effect of the changing covariance matrix $\Sigma$ on optimal weights.

## Step 5.1 — Equal Risk Contribution

In [ ]:
N      = returns.shape[1]
w0     = np.ones(N) / N
bounds = tuple((0.0, 0.70) for _ in range(N))
cons_sum = {'type': 'eq', 'fun': lambda w: w.sum() - 1.0}

def erc_objective(w, cov):
    port_vol = np.sqrt(w @ cov @ w)
    mrc = (cov @ w) / port_vol
    rc  = w * mrc
    return ((rc - rc.mean()) ** 2).sum()

cov_pre_erc  = ret_pre.cov().values
cov_post_erc = ret_post.cov().values

w_erc_pre  = minimize(erc_objective, w0, args=(cov_pre_erc,),
                      bounds=bounds, constraints=cons_sum).x
w_erc_post = minimize(erc_objective, w0, args=(cov_post_erc,),
                      bounds=bounds, constraints=cons_sum).x

print("ERC weights — Pre vs Post COVID:")
print(pd.DataFrame({'Pre-COVID': w_erc_pre, 'Post-COVID': w_erc_post},
                   index=returns.columns).round(3))

## Step 5.2 — Markowitz with Regime-Conditional Covariance

We use the smoothed regime probabilities from the Gold-vs-SP500 Markov switching
model to classify each day into regime 1 (good hedge, $P(S_t=2) < 0.3$) or
regime 2 (bad hedge, $P(S_t=2) > 0.7$). Days in between are excluded to avoid
mixing regimes. The covariance matrix is then estimated separately from each
regime's return history. Expected returns $\mu$ are the unconditional full-sample
means — held constant to isolate the covariance effect on optimal allocation.

In [ ]:
try:
    prob_r2 = ms_full['Gold'].smoothed_marginal_probabilities.iloc[:, 1]
except Exception:
    prob_r2 = ms_full['Gold'].smoothed_marginal_probabilities[1]

r1_dates = prob_r2[prob_r2 < 0.3].index
r2_dates = prob_r2[prob_r2 > 0.7].index

cov_r1 = returns.loc[r1_dates].cov().values
cov_r2 = returns.loc[r2_dates].cov().values

mu_full    = returns.mean().values
target_ret = mu_full @ w0

cons_mark = [
    {'type': 'eq', 'fun': lambda w: w.sum() - 1.0},
    {'type': 'eq', 'fun': lambda w: w @ mu_full - target_ret},
]

def markowitz_objective(w, cov):
    return w @ cov @ w

w_mark_r1 = minimize(markowitz_objective, w0, args=(cov_r1,),
                     bounds=bounds, constraints=cons_mark).x
w_mark_r2 = minimize(markowitz_objective, w0, args=(cov_r2,),
                     bounds=bounds, constraints=cons_mark).x

# ── Also compute pre/post Markowitz to show cost of not updating ───────────────
w_mark_pre  = minimize(markowitz_objective, w0, args=(cov_pre_erc * 252,),
                       bounds=bounds, constraints=cons_mark).x
w_mark_post = minimize(markowitz_objective, w0, args=(cov_post_erc * 252,),
                       bounds=bounds, constraints=cons_mark).x

cov_pre_ann  = cov_pre_erc  * 252
cov_post_ann = cov_post_erc * 252
vol_pre_in   = np.sqrt(w_mark_pre  @ cov_pre_ann  @ w_mark_pre)
vol_post_in  = np.sqrt(w_mark_post @ cov_post_ann @ w_mark_post)
vol_mismatch = np.sqrt(w_mark_pre  @ cov_post_ann @ w_mark_pre)

print(f"Portfolio vol (pre weights × pre Σ):   {vol_pre_in*100:.2f}%")
print(f"Portfolio vol (post weights × post Σ): {vol_post_in*100:.2f}%")
print(f"Portfolio vol (pre weights × post Σ):  {vol_mismatch*100:.2f}%  ← cost of inaction")
print(f"Extra annualised vol from not updating: +{(vol_mismatch-vol_post_in)*100:.2f}%")

## Step 5.3 — Performance Metrics & Summary Table

In [ ]:
def perf_metrics(w, returns_period):
    port   = returns_period.values @ w
    mean_a = port.mean() * 252
    vol_a  = port.std()  * np.sqrt(252)
    sharpe = mean_a / vol_a if vol_a > 0 else np.nan
    eq     = (1 + port / 100).cumprod()
    mdd    = (eq / np.maximum.accumulate(eq) - 1).min()
    dr     = (w * returns_period.std().values * np.sqrt(252)).sum() / vol_a
    return {'Sharpe': round(sharpe, 3), 'MDD %': round(mdd * 100, 2),
            'Div Ratio': round(dr, 3)}

rows = []
for label, w, ret_sub in [
    ('ERC pre-COVID',       w_erc_pre,  ret_pre),
    ('ERC post-COVID',      w_erc_post, ret_post),
    ('Markowitz regime 1',  w_mark_r1,  returns.loc[r1_dates]),
    ('Markowitz regime 2',  w_mark_r2,  returns.loc[r2_dates]),
]:
    m = perf_metrics(w, ret_sub)
    m['Strategy'] = label
    rows.append(m)

table5_perf = pd.DataFrame(rows).set_index('Strategy')
table5_weights = pd.DataFrame({
    'ERC pre':       w_erc_pre,
    'ERC post':      w_erc_post,
    'MV regime 1':   w_mark_r1,
    'MV regime 2':   w_mark_r2,
}, index=returns.columns).round(3)

print("Weights:")
display(table5_weights)
print("\nPerformance:")
display(table5_perf)

table5_weights.to_csv('outputs/tables/table5_weights.csv')
table5_perf.to_csv('outputs/tables/table5_perf.csv')

# ── Figure: weight comparison bar chart ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, cols) in zip(axes, [
    ('ERC: Pre vs Post COVID',
     {'Pre-COVID': w_erc_pre, 'Post-COVID': w_erc_post}),
    ('Markowitz: Regime 1 (hedge) vs Regime 2 (breakdown)',
     {'Regime 1': w_mark_r1, 'Regime 2': w_mark_r2}),
]):
    x     = np.arange(len(returns.columns))
    width = 0.35
    for i, (label, w) in enumerate(cols.items()):
        color = 'steelblue' if i == 0 else 'firebrick'
        ax.bar(x + (i - 0.5) * width, w, width, label=label, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(returns.columns, rotation=20, fontsize=8)
    ax.set_ylabel('Portfolio weight')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Portfolio Weights: Effect of COVID and Regime on Optimal Allocation\n'
             '(μ held constant — isolates pure covariance effect)', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/figures/portfolio_weights.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 6. Results Summary

Consolidated table: sup-F break detection, Bai-Perron break dates, Markov switching
expected durations, and Wald test verdict — all eight safe-haven / benchmark pairs.

In [ ]:
summary = []
for key in PAIRS:
    sh, bm = key.split('_vs_')
    sup    = supF_results.get(key, {})
    bp     = bp_results.get(key, {})
    wd     = wald_results.get(sh, {}) if bm == 'SP500' else {}
    bd_str = ', '.join(str(d.date()) for d in bp.get('break_dates', [])) or 'None'

    summary.append({
        'Pair':            key,
        'sup-F stat':      round(sup.get('sup_F', np.nan), 2),
        'Break detected':  'Yes' if sup.get('reject') else 'No',
        'BP breaks (m)':   bp.get('optimal_m', 0),
        'Break date(s)':   bd_str,
        'Wald p':          round(wd.get('p', np.nan), 4),
        'E[D_bad] pre (d)':  round(wd.get('D_bad_pre',  np.nan), 1),
        'E[D_bad] post (d)': round(wd.get('D_bad_post', np.nan), 1),
    })

summary_df = pd.DataFrame(summary)
print("\n" + "="*80 + "\nFINAL RESULTS SUMMARY\n" + "="*80)
display(summary_df.set_index('Pair'))
summary_df.to_csv('outputs/tables/results_summary.csv', index=False)
print("\nAll outputs saved to outputs/tables/ and outputs/figures/")